In [ ]:
from qiskit import QuantumRegister, ClassicalRegister, QuantumCircuit, transpile, assemble
import numpy as np
from qiskit import execute, Aer, IBMQ
from qiskit.providers.aer.noise import NoiseModel
import qiskit.circuit.library
import qiskit.quantum_info
from qiskit.tools.monitor import job_monitor
from qiskit.visualization import plot_histogram, plot_bloch_multivector
from qiskit.providers.aer import extensions 
from qiskit.providers.aer.noise import NoiseModel
import qiskit.providers.aer.noise as noise
import numpy as np
import time
from copy import deepcopy
import qiskit.quantum_info as qi
from qiskit.compiler import assemble
from qiskit.ignis.verification.tomography import state_tomography_circuits, StateTomographyFitter
from qiskit.ignis.verification.tomography import process_tomography_circuits, ProcessTomographyFitter
from qiskit.ignis.verification.tomography import gateset_tomography_circuits, GatesetTomographyFitter
import qiskit.ignis.mitigation.measurement as mc
from qiskit.quantum_info import Choi, Kraus
from qiskit.extensions import HGate, XGate
from qiskit.aqua import QuantumInstance
import mitiq
from mitiq.zne.scaling import fold_global, fold_gates_at_random
from qiskit.circuit.random import random_circuit
import matplotlib.pyplot as plt
import scipy

In [ ]:
# create ZNE function

# method is string to choose extrapoaliton scheme R - richardson (default), P - polynomial, PE - polyexponential

def ZNE(circuit, order, backend, s, method = "R", a = 0):
    y = []
    mitigated = 0
    circuit = qiskit.compiler.transpile(circuit, basis_gates=["u1", "u2", "u3", "cx"], optimization_level=0)
    scaling = np.linspace(1.0, order,  order)
    #print(scaling)
    for i in scaling:
        folded = fold_gates_at_random(circuit, scale_factor=i)
        folded.measure_all()
        result = execute(folded, backend).result() #runscaled circuit
        counts = result.get_counts() #counts scaled circuit
        if counts.get("0") is None:
            expectation = 0.
        else:
            expectation = counts.get("0") / s
        y = np.append(y, expectation)
    if method == "R":
        for k in range(0, len(y)):
            product = 1
            for i in range(0, len(y)):
                if k != i:
                    product = product * (scaling[i]/(scaling[i]-scaling[k]))
                    print(product)
            mitigated = mitigated + y[k]*product
    elif method == "P":
        z = np.polyfit(scaling, y, (order-1))
        f = np.poly1d(z)
        mitigated= f(0)
        mitigated = a + np.exp(mitigated)
    elif method == "PE":
        y = y-a
        y = np.log(y)
        z = np.polyfit(scaling, y, (order-1))
        f = np.poly1d(z)
        mitigated= f(0)
        mitigated = a + np.exp(mitigated)
   # elif method == "E":
    #    if order == 2:
     #       r0 = scaling[1]/(scaling[1]-1)
      #      r1 = 1/(1-scaling[1])
        #    print(r0)
       #     print(r1)
     #       mitigated = y[0]**r0 + y[1]**r1
      #  else:
      #      mitigated = 0.
      #      print("Exponential extrapolation scheme can only be used to 2 order")
    else:
        print("Provide with valid extrapoaltion scheme R, PE, P")
    circuit.measure_all()
    result = execute(circuit, Aer.get_backend('qasm_simulator', shots = s)).result() #runscaled circuit
    counts = result.get_counts() #counts scaled circuit
    if counts.get("0") is None:
        ideal = 0.
    else:
        ideal = counts.get("0") / s
    return ideal, mitigated

In [ ]:
# test 
N = 1
qreg_q = QuantumRegister(N, 'q')
creg_c = ClassicalRegister(N, 'c')
circuit = QuantumCircuit(N)
circuit.h(0)
backend = Aer.get_backend('aer_simulator', shots=8000)

a, b = ZNE(circuit, 2, backend, 8000, method = "R", a = 0)
print(a)
print(b)